In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.style.use('dark_background' )
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from tensorflow.keras.utils import to_categorical, plot_model
import os

In [ ]:
import os
from PIL import Image
import numpy as np

data = []
result = []

folders = [
    (r"C:\Users\LOQ\OneDrive\Desktop\Deep Learning\Brain Tumor\brain_tumor_dataset\glioma", 0),
    (r"C:\Users\LOQ\OneDrive\Desktop\Deep Learning\Brain Tumor\brain_tumor_dataset\healthy", 1),
    (r"C:\Users\LOQ\OneDrive\Desktop\Deep Learning\Brain Tumor\brain_tumor_dataset\meningioma", 2),
    (r"C:\Users\LOQ\OneDrive\Desktop\Deep Learning\Brain Tumor\brain_tumor_dataset\pituitary", 3)
]

for folder_path, label in folders:
    for r, d, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                try:
                    path = os.path.join(r, file)

                    img = Image.open(path).convert('RGB')   # force 3 channels
                    img = img.resize((128, 128))
                    img = np.array(img) / 255.0            # normalization

                    if img.shape == (128, 128, 3):
                        data.append(img)
                        result.append(label)

                except:
                    continue  # skip corrupted images

# ✅ convert AFTER loop
data = np.array(data)
result = np.array(result)

# ✅ sanity check
print("Data shape:", data.shape)
print("Label shape:", result.shape)

In [ ]:
plt.imshow(data[5])

In [ ]:
plt.imshow(data[400])

In [ ]:
plt.imshow(data[5000])

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    data, result, test_size=0.2, shuffle=True, random_state=42
)

In [ ]:
model = Sequential()

# Block 1
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3), padding='same'))
model.add(Conv2D(32, (3,3), activation='relu', padding='same'))
model.add(MaxPooling2D((2,2)))
model.add(BatchNormalization())

# Block 2
model.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model.add(MaxPooling2D((2,2)))
model.add(BatchNormalization())

# Block 3
model.add(Conv2D(128, (3,3), activation='relu', padding='same'))
model.add(MaxPooling2D((2,2)))

# Dense
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))

# Output
model.add(Dense(4, activation='softmax'))

print(model.summary())

In [ ]:
model.compile(loss="sparse_categorical_crossentropy",optimizer="adam",metrics=["accuracy"])

In [ ]:
x_train.shape

In [ ]:
y_train.shape

In [ ]:
history = model.fit(x_train, y_train,epochs=10,batch_size=64,verbose=1,validation_data=(x_test, y_test))

In [ ]:
score = model.evaluate(x_test, y_test, batch_size=64)
print("\nTest accuracy: %.1f%%" % (100.0 * score[1]))

In [ ]:
#manual prediction
img = Image.open(r"C:\Users\LOQ\OneDrive\Desktop\Deep Learning\Brain Tumor\brain_tumor_dataset\glioma\0352.jpg").convert('RGB')
img = img.resize((128, 128))
img_array = np.array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)  # add batch dimension
pred = model.predict(img_array)
predicted_class = np.argmax(pred)
if predicted_class == 0:
    print("Predicted: Glioma")
elif predicted_class == 1:
    print("Predicted: Healthy")
elif predicted_class == 2:
    print("Predicted: Meningioma")
elif predicted_class == 3:
    print("Predicted: Pituitary")


In [ ]:
#Testing the model with a new images


classes = ["glioma", "healthy", "meningioma", "pituitary"]

# 🔹 PASTE YOUR IMAGE PATH HERE
img_path = r"C:\Users\LOQ\OneDrive\Desktop\Deep Learning\Brain Tumor\brain_tumor_dataset\Test\meningioma_tumor4.jpg"
# load image
img = Image.open(img_path).convert('RGB')
img = img.resize((128, 128))
img = np.array(img) / 255.0

# reshape
img = np.expand_dims(img, axis=0)

# predict
pred = model.predict(img, verbose=0)

class_index = np.argmax(pred)
confidence = np.max(pred)

# result
print("Prediction:", classes[class_index])
print("Confidence:", round(confidence * 100, 2), "%")

In [ ]:
#Testing the model with a new images


classes = ["glioma", "healthy", "meningioma", "pituitary"]

# 🔹 PASTE YOUR IMAGE PATH HERE
img_path = r"C:\Users\LOQ\OneDrive\Desktop\Deep Learning\Brain Tumor\brain_tumor_dataset\Test\meningioma_tumor4.jpg"
# load image
img = Image.open(img_path).convert('RGB')
img = img.resize((128, 128))
img = np.array(img) / 255.0

# reshape
img = np.expand_dims(img, axis=0)

# predict
pred = model.predict(img, verbose=0)

class_index = np.argmax(pred)
confidence = np.max(pred)

# result
print("Prediction:", classes[class_index])
print("Confidence:", round(confidence * 100, 2), "%")

In [ ]:
#save the model in the native Keras format
model.save('brain_tumor_model.h5')

# To this:
model.save('brain_tumor_model.keras')